# Tesis - MDM UBA - 2026

**Tariff classification using NLP**

# Baselines iterations

New enviroment is needed for replication of doc2vec baseline

**doc2vec** & **fasttext** library

!pip install gensim==4.3.3

In [1]:
# Gral dependencies
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm import tqdm
tqdm.pandas()
from datetime import datetime
import re
from typing import Iterable, List, Tuple, Union, Optional, Dict, Any
from collections import defaultdict
import matplotlib.pyplot as plt

### Raw dataset

In [2]:
colspecs = [(0, 6), (6, None)]
data_type = {'HS06': str}
df = pd.read_fwf('data/raw_data_HScodes_desc.txt',
                 colspecs=colspecs, header=None,
                 names=['HS06', 'GOODS_DESCRIPTION'],
                 dtype=data_type)

### Quick EDA

null and duplicated samples

dropping duplicates

analyzing tops and bottoms regarding frequencies

In [3]:
# Quick EDA
print("=== Quick EDA ===")

# Add HS02 (chapter) and HS04 (heading)
df['HS04'] = df['HS06'].str[:4]
df['HS02'] = df['HS06'].str[:2]

print("Nulls per column:")
print(df.isnull().sum(), "")

print("Duplicate rows:", df.duplicated().sum(), "")

# Function to build and display freq tables
def freq_table(col, name):
    vc      = df[col].value_counts().rename('count')
    rel     = df[col].value_counts(normalize=True).rename('rel_freq')
    cum     = rel.cumsum().rename('cum_freq')
    summary = pd.concat([vc, rel, cum], axis=1)
    summary['rel_freq'] = (summary['rel_freq'] * 100).round(2).astype(str) + '%'
    summary['cum_freq'] = (summary['cum_freq'] * 100).round(2).astype(str) + '%'

    print(f"## Samples per {name} ({col})")
    print("### Top 10")
    print(summary.head(10).to_markdown(), "\n")
    print("### Bottom 10")
    print(summary.tail(10).to_markdown(), "\n")

# Dropping duplicates
df.drop_duplicates(inplace=True)

# Chapter-level (HS02)
freq_table('HS02', 'chapter')

# Heading-level (HS04)
freq_table('HS04', 'heading')

# Subheading-level (HS06)
freq_table('HS06', 'subheading')

=== Quick EDA ===
Nulls per column:
HS06                 0
GOODS_DESCRIPTION    0
HS04                 0
HS02                 0
dtype: int64 
Duplicate rows: 232220 
## Samples per chapter (HS02)
### Top 10
|   HS02 |   count | rel_freq   | cum_freq   |
|-------:|--------:|:-----------|:-----------|
|     84 |   54901 | 20.5%      | 20.5%      |
|     85 |   33571 | 12.54%     | 33.04%     |
|     87 |   28476 | 10.63%     | 43.67%     |
|     73 |   16173 | 6.04%      | 49.71%     |
|     39 |   12218 | 4.56%      | 54.28%     |
|     90 |   11611 | 4.34%      | 58.61%     |
|     82 |    7972 | 2.98%      | 61.59%     |
|     94 |    7921 | 2.96%      | 64.55%     |
|     40 |    7526 | 2.81%      | 67.36%     |
|     83 |    4285 | 1.6%       | 68.96%     | 

### Bottom 10
|   HS02 |   count | rel_freq   | cum_freq   |
|-------:|--------:|:-----------|:-----------|
|     41 |      22 | 0.01%      | 99.96%     |
|     81 |      19 | 0.01%      | 99.97%     |
|     45 |      19 | 0.01

### Preprocessing of text

In [4]:
stop_words = {'of', 'or', 'and', 'for', 'than', 'the', 'in', 'with', 'to', 'but', 'by'
             , 'whether', 'on', 'its', 'an', 'their', 'at', 'this', 'which', 'from'
             , 'as', 'be', 'is'}
alphabet_pattern = re.compile(r'[^a-zA-Z]')
alphabet_number_pattern = re.compile(r'[^a-zA-Z0-9]')
remove_pattern = re.compile(r'[\;\,\)\(\[\]\:]')


def refine_text_func(text):
    text = text.lower()
    text = ' '.join([w for w in text.split() if w not in stop_words])
    alphabet = re.sub(alphabet_pattern, ' ', text)
    alphabet_number = re.sub(alphabet_number_pattern, ' ', text)
    remove = re.sub(remove_pattern, ' ', text)
    result = ' '.join([text, alphabet, alphabet_number, remove])
    return result

In [5]:
df['PREPRO_DESCRIPTION'] = df['GOODS_DESCRIPTION'].progress_apply(lambda x: refine_text_func(x))

100%|██████████| 267780/267780 [00:01<00:00, 190012.56it/s]


### N-gram generation

In [6]:
def create_ngram_data(text, ngram_value=2):
    text_list = text.split()
    ngram_list = list(zip(*[text_list[i:] for i in range(ngram_value)]))
    result = []
    for n_data in ngram_list:
        result.append('_'.join(n_data))
    return ' '.join(result)

create_ngram_data('LIVE BREEDING FARM HORSE')

'LIVE_BREEDING BREEDING_FARM FARM_HORSE'

In [7]:
df['NGRAM_DESCRIPTION'] = df['PREPRO_DESCRIPTION'].progress_apply(lambda x: create_ngram_data(x))

100%|██████████| 267780/267780 [00:00<00:00, 271352.29it/s]


In [8]:
df.head()

,HS06,GOODS_DESCRIPTION,HS04,HS02,PREPRO_DESCRIPTION,NGRAM_DESCRIPTION
0,271019,BRAKE FLUID DOT 4 50X200ML,2710,27,brake fluid dot 4 50x200ml brake fluid dot ...,brake_fluid fluid_dot dot_4 4_50x200ml 50x200m...
1,847710,PLASTIC INJECTION MOULD MODEL 21A 110G DSM1010...,8477,84,plastic injection mould model 21a 110g dsm1010...,plastic_injection injection_mould mould_model ...
2,844399,LCD ASSEMBLY,8443,84,lcd assembly lcd assembly lcd assembly lcd ass...,lcd_assembly assembly_lcd lcd_assembly assembl...
3,848280,BEARING 22238 KCAW33C3 BRAND MCB,8482,84,bearing 22238 kcaw33c3 brand mcb bearing ...,bearing_22238 22238_kcaw33c3 kcaw33c3_brand br...
4,630900,USED HANDBAGS AND WALLETS,6309,63,used handbags wallets used handbags wallets us...,used_handbags handbags_wallets wallets_used us...


Sampling function

In [9]:
def bootstrap_sampling(df, test_fraction=0.1, seed=32):
    # Determine the number of test samples
    n_test = int(len(df) * test_fraction)
    # Perform bootstrap sampling for the test set
    test_set = df.sample(n=n_test, replace=True, random_state=seed)
    # Remove the test samples from the original dataframe to create the training set
    train_set = df.drop(test_set.index)
    
    return train_set, test_set

## Iterations definitions

In [10]:
import random
import joblib

fraction = 0.05
iterations = 10

min_val = 0
max_val = 999999999
random_seed = random.randint(min_val, max_val)

seeds = []

for iter in range(iterations):
    seed = random.randint(min_val, max_val)
    seeds.append(seed)

print("Random seeds for each iteration:")
print(seeds)  

out_dir = "results/baselines"
os.makedirs(out_dir, exist_ok=True)

Random seeds for each iteration:
[226944881, 768593320, 366261559, 531210901, 715275432, 908272322, 173892995, 625333247, 370247453, 503070489]


### Doc2Vec

In [11]:
from gensim.models import Doc2Vec

### Evaluating Doc2Vec

Evaluation function

In [12]:
from doc2vec_utils import evaluate_df_d2v

### FastText

FastText configuration to be used as Doc2Vec

In [13]:
from fasttext_utils import FastTextDocVec

### Evaluating FastText

Evaluation function

In [14]:
from fasttext_utils import evaluate_df_ft

In [15]:
# df = df.sample(frac=0.1)

## Training a Doc2Vec as baseline

A- Raw descriptions

B- Preproced descriptions

C- Preproced + N-gram descriptions

In [16]:
# Model dependences
from gensim.models.doc2vec import TaggedDocument

target_col = 'HS04'
window = 5 # context window size +/- 
num_epochs = 50
model_dim = 254
seed = 32

raw_col = 'GOODS_DESCRIPTION'
prepro_col = 'PREPRO_DESCRIPTION'
ngram_col = 'NGRAM_DESCRIPTION'

#### A- Raw descriptions

In [17]:
all_metrics = []
scored_dfs = {}

text_col = raw_col 

for iter in range(iterations):
    seed = seeds[iter]
    print(f"\n=== Iteration {iter+1}/{iterations} seed {seed} ===")
    model_name = f"D2V_{text_col}_{target_col}_seed{seed}"
    print(f"Model name: {model_name}")

    train_df, val_df = bootstrap_sampling(df, test_fraction=fraction, seed=seed)

    print("Training Doc2Vec model")
    model = Doc2Vec(window=window, 
                min_count=1, # ignore 1 instead of not ignore any words
                vector_size=model_dim, # dim of the feature vectors
                sample=1e-4, # threshold randomly down-sample high-frequency words
                hs=1, # hierarchical softmax instead of negative sampling
                max_vocab_size=None, # no limit
                alpha=0.025, # initial learning rate
                min_alpha=0.001, # min learning rate
                dm=0, # PV-DBOW 
                dbow_words=0, # only trains doc-vectors
                dm_tag_count=1, # one tag per document
                dm_mean=0, # use the sum of the context word vectors
                dm_concat=0, # for smaller model
                negative=5, # number of negative samples
                seed=seed, # random seed
                workers=os.cpu_count()
                )
    
    print("Building vocabulary")

    sentences = []
    for idx, row in tqdm(train_df.iterrows(), total=train_df.shape[0]):
        words_list = row[text_col].split()
        sentences.append(TaggedDocument(words_list, [row[target_col]]))

    print(sentences[:3])

    model.build_vocab(sentences)

    print("Training model")
    model.train(sentences, total_examples=model.corpus_count, epochs=num_epochs)

    print("Evaluating model")

    df_scored, metrics = evaluate_df_d2v(
        val_df,
        model=model,
        model_name=model_name,
        text_col=text_col,
        target_col=target_col,
        top_n=5,
        epochs=num_epochs,           # matches your trained-model naming
        alpha=None,          # let gensim handle the schedule
        min_alpha=None,
        show_progress=True
    )

    scored_dfs[model_name] = df_scored
    row = {'model': model_name, **metrics}
    all_metrics.append(row)

    del model

print("=== Metrics summary ===")
metrics_df = pd.DataFrame(all_metrics).set_index('model').sort_index()
display(metrics_df)
display(metrics_df.describe())

metrics_df.to_csv(os.path.join(out_dir, f"scored_dfs_{model_name}.csv"), index=False)
print(f"Saved metrics to {os.path.join(out_dir, f'scored_dfs_{model_name}.csv')}")

joblib.dump(
    scored_dfs,
    os.path.join(out_dir, f"scored_dfs_{model_name}.joblib"),
    compress=3
)
print(f"Saved scored_dfs to {os.path.join(out_dir, f'scored_dfs_{model_name}.joblib')}")


=== Iteration 1/10 seed 226944881 ===
Model name: D2V_GOODS_DESCRIPTION_HS04_seed226944881
Training Doc2Vec model
Building vocabulary


100%|██████████| 254721/254721 [00:04<00:00, 52329.03it/s]


[TaggedDocument(words=['BRAKE', 'FLUID', 'DOT', '4', '50X200ML'], tags=['2710']), TaggedDocument(words=['LCD', 'ASSEMBLY'], tags=['8443']), TaggedDocument(words=['BEARING', '22238', 'KCAW33C3', 'BRAND', 'MCB'], tags=['8482'])]
Training model
Evaluating model
Model: D2V_GOODS_DESCRIPTION_HS04_seed226944881
Text column: GOODS_DESCRIPTION
Target column: HS04
Top-N: 5


100%|██████████| 13389/13389 [00:14<00:00, 896.03it/s]


Total samples: 13389
Top-1 Accuracy: 0.5142 (6884/13389)
Top-2 Accuracy: 0.6048 (8098/13389)
Top-3 Accuracy: 0.6496 (8696/13389)
Top-4 Accuracy: 0.6784 (9083/13389)
Top-5 Accuracy: 0.6977 (9340/13389)

=== Iteration 2/10 seed 768593320 ===
Model name: D2V_GOODS_DESCRIPTION_HS04_seed768593320
Training Doc2Vec model
Building vocabulary


...

100%|██████████| 13389/13389 [00:14<00:00, 924.92it/s]


Total samples: 13389
Top-1 Accuracy: 0.5101 (6829/13389)
Top-2 Accuracy: 0.6003 (8037/13389)
Top-3 Accuracy: 0.6462 (8651/13389)
Top-4 Accuracy: 0.6711 (8984/13389)
Top-5 Accuracy: 0.6894 (9231/13389)
=== Metrics summary ===


,top_1_acc,top_2_acc,top_3_acc,top_4_acc,top_5_acc
model,,,,,
D2V_GOODS_DESCRIPTION_HS04_seed173892995,0.519083,0.613339,0.654791,0.682127,0.700799
D2V_GOODS_DESCRIPTION_HS04_seed226944881,0.514228,0.604825,0.649563,0.678393,0.697662
D2V_GOODS_DESCRIPTION_HS04_seed366261559,0.513033,0.605945,0.647920,0.675704,0.694675
D2V_GOODS_DESCRIPTION_HS04_seed370247453,0.510643,0.604601,0.650235,0.677496,0.695422
D2V_GOODS_DESCRIPTION_HS04_seed503070489,0.510120,0.600269,0.646202,0.671073,0.689447
D2V_GOODS_DESCRIPTION_HS04_seed531210901,0.514377,0.602136,0.645605,0.673837,0.692658
D2V_GOODS_DESCRIPTION_HS04_seed625333247,0.501456,0.592352,0.641721,0.666965,0.687654
D2V_GOODS_DESCRIPTION_HS04_seed715275432,0.511913,0.605721,0.647472,0.672567,0.689745
D2V_GOODS_DESCRIPTION_HS04_seed768593320,0.515124,0.609456,0.652028,0.678243,0.694600


,top_1_acc,top_2_acc,top_3_acc,top_4_acc,top_5_acc
count,10.000000,10.000000,10.000000,10.000000,10.000000
mean,0.512032,0.604354,0.648555,0.675196,0.693719
std,0.004606,0.005552,0.003643,0.004304,0.003984
min,0.501456,0.592352,0.641721,0.666965,0.687654
25%,0.510419,0.602752,0.646520,0.672884,0.690473
50%,0.512473,0.604862,0.648742,0.675629,0.694562
75%,0.514340,0.605889,0.650179,0.678056,0.695235
max,0.519083,0.613339,0.654791,0.682127,0.700799


Saved metrics to results/baselines\scored_dfs_D2V_GOODS_DESCRIPTION_HS04_seed503070489.csv
Saved scored_dfs to results/baselines\scored_dfs_D2V_GOODS_DESCRIPTION_HS04_seed503070489.joblib


took 49 min

In [18]:
#scored_dfs = joblib.load(os.path.join(out_dir, f"scored_dfs_{model_name}.joblib"))

#### B- Preproced descriptions

In [19]:
all_metrics = []
scored_dfs = {}

text_col = prepro_col 

for iter in range(iterations):
    seed = seeds[iter]
    print(f"\n=== Iteration {iter+1}/{iterations} seed {seed} ===")
    model_name = f"D2V_{text_col}_{target_col}_seed{seed}"
    print(f"Model name: {model_name}")

    train_df, val_df = bootstrap_sampling(df, test_fraction=fraction, seed=seed)

    print("Training Doc2Vec model")
    model = Doc2Vec(window=window, 
                    min_count=1, # ignore 1 instead of not ignore any words
                    vector_size=model_dim, # dim of the feature vectors
                    sample=1e-4, # threshold randomly down-sample high-frequency words
                    hs=1, # hierarchical softmax instead of negative sampling
                    max_vocab_size=None, # no limit
                    alpha=0.025, # initial learning rate
                    min_alpha=0.001, # min learning rate
                    dm=0, # PV-DBOW 
                    dbow_words=0, # only trains doc-vectors
                    dm_tag_count=1, # one tag per document
                    dm_mean=0, # use the sum of the context word vectors
                    dm_concat=0, # for smaller model
                    negative=5, # number of negative samples
                    seed=seed, # random seed
                    workers=os.cpu_count()
                    )
    
    print("Building vocabulary")

    sentences = []
    for idx, row in tqdm(train_df.iterrows(), total=train_df.shape[0]):
        words_list = row[text_col].split()
        sentences.append(TaggedDocument(words_list, [row[target_col]]))

    print(sentences[:3])

    model.build_vocab(sentences)

    print("Training model")
    model.train(sentences, total_examples=model.corpus_count, epochs=num_epochs)

    print("Evaluating model")

    df_scored, metrics = evaluate_df_d2v(
        val_df,
        model=model,
        model_name=model_name,
        text_col=text_col,
        target_col=target_col,
        top_n=5,
        epochs=num_epochs,           # matches your trained-model naming
        alpha=None,          # let gensim handle the schedule
        min_alpha=None,
        show_progress=True
    )

    scored_dfs[model_name] = df_scored
    row = {'model': model_name, **metrics}
    all_metrics.append(row)

    del model

print("=== Metrics summary ===")
metrics_df = pd.DataFrame(all_metrics).set_index('model').sort_index()
display(metrics_df)
display(metrics_df.describe())

metrics_df.to_csv(os.path.join(out_dir, f"scored_dfs_{model_name}.csv"), index=False)
print(f"Saved metrics to {os.path.join(out_dir, f'scored_dfs_{model_name}.csv')}")

joblib.dump(
    scored_dfs,
    os.path.join(out_dir, f"scored_dfs_{model_name}.joblib"),
    compress=3
)
print(f"Saved scored_dfs to {os.path.join(out_dir, f'scored_dfs_{model_name}.joblib')}")


=== Iteration 1/10 seed 226944881 ===
Model name: D2V_PREPRO_DESCRIPTION_HS04_seed226944881
Training Doc2Vec model
Building vocabulary


100%|██████████| 254721/254721 [00:05<00:00, 50779.90it/s]


[TaggedDocument(words=['brake', 'fluid', 'dot', '4', '50x200ml', 'brake', 'fluid', 'dot', 'x', 'ml', 'brake', 'fluid', 'dot', '4', '50x200ml', 'brake', 'fluid', 'dot', '4', '50x200ml'], tags=['2710']), TaggedDocument(words=['lcd', 'assembly', 'lcd', 'assembly', 'lcd', 'assembly', 'lcd', 'assembly'], tags=['8443']), TaggedDocument(words=['bearing', '22238', 'kcaw33c3', 'brand', 'mcb', 'bearing', 'kcaw', 'c', 'brand', 'mcb', 'bearing', '22238', 'kcaw33c3', 'brand', 'mcb', 'bearing', '22238', 'kcaw33c3', 'brand', 'mcb'], tags=['8482'])]
Training model
Evaluating model
Model: D2V_PREPRO_DESCRIPTION_HS04_seed226944881
Text column: PREPRO_DESCRIPTION
Target column: HS04
Top-N: 5


100%|██████████| 13389/13389 [00:37<00:00, 359.19it/s]


Total samples: 13389
Top-1 Accuracy: 0.4775 (6393/13389)
Top-2 Accuracy: 0.5591 (7486/13389)
Top-3 Accuracy: 0.5969 (7992/13389)
Top-4 Accuracy: 0.6244 (8359/13389)
Top-5 Accuracy: 0.6427 (8605/13389)

=== Iteration 2/10 seed 768593320 ===
Model name: D2V_PREPRO_DESCRIPTION_HS04_seed768593320
Training Doc2Vec model
Building vocabulary


...

100%|██████████| 13389/13389 [00:43<00:00, 307.96it/s]


Total samples: 13389
Top-1 Accuracy: 0.4707 (6302/13389)
Top-2 Accuracy: 0.5532 (7406/13389)
Top-3 Accuracy: 0.5960 (7980/13389)
Top-4 Accuracy: 0.6197 (8297/13389)
Top-5 Accuracy: 0.6377 (8538/13389)
=== Metrics summary ===


,top_1_acc,top_2_acc,top_3_acc,top_4_acc,top_5_acc
model,,,,,
D2V_PREPRO_DESCRIPTION_HS04_seed173892995,0.483606,0.564344,0.604003,0.628725,0.648742
D2V_PREPRO_DESCRIPTION_HS04_seed226944881,0.477482,0.559116,0.596908,0.624393,0.642692
D2V_PREPRO_DESCRIPTION_HS04_seed366261559,0.477407,0.558145,0.598924,0.624468,0.642244
D2V_PREPRO_DESCRIPTION_HS04_seed370247453,0.469714,0.551946,0.592501,0.617522,0.636567
D2V_PREPRO_DESCRIPTION_HS04_seed503070489,0.470685,0.553215,0.596012,0.619688,0.637688
D2V_PREPRO_DESCRIPTION_HS04_seed531210901,0.473971,0.559265,0.598775,0.624468,0.644335
D2V_PREPRO_DESCRIPTION_HS04_seed625333247,0.463664,0.543954,0.586153,0.612592,0.630518
D2V_PREPRO_DESCRIPTION_HS04_seed715275432,0.473598,0.556950,0.593696,0.619165,0.637613
D2V_PREPRO_DESCRIPTION_HS04_seed768593320,0.479349,0.562551,0.601240,0.628426,0.645381


,top_1_acc,top_2_acc,top_3_acc,top_4_acc,top_5_acc
count,10.000000,10.000000,10.000000,10.000000,10.000000
mean,0.474457,0.556785,0.596833,0.622533,0.640937
std,0.005579,0.005842,0.005090,0.005149,0.005305
min,0.463664,0.543954,0.586153,0.612592,0.630518
25%,0.471413,0.554149,0.594275,0.619296,0.637632
50%,0.474531,0.558257,0.597841,0.624430,0.642468
75%,0.477463,0.559228,0.599821,0.625532,0.644148
max,0.483606,0.564344,0.604003,0.628725,0.648742


Saved metrics to results/baselines\scored_dfs_D2V_PREPRO_DESCRIPTION_HS04_seed503070489.csv
Saved scored_dfs to results/baselines\scored_dfs_D2V_PREPRO_DESCRIPTION_HS04_seed503070489.joblib


Took 74 min

#### C- Preproced + N-gram descriptions

In [20]:
all_metrics = []
scored_dfs = {}

text_col = prepro_col +'_'+ ngram_col
df[text_col] = df[prepro_col] + ' ' + df[ngram_col]

for iter in range(iterations):
    seed = seeds[iter]
    print(f"\n=== Iteration {iter+1}/{iterations} seed {seed} ===")
    model_name = f"D2V_{text_col}_{target_col}_seed{seed}"
    print(f"Model name: {model_name}")

    train_df, val_df = bootstrap_sampling(df, test_fraction=fraction, seed=seed)

    print("Training Doc2Vec model")
    model = Doc2Vec(window=window, 
                    min_count=1, # ignore 1 instead of not ignore any words
                    vector_size=model_dim, # dim of the feature vectors
                    sample=1e-4, # threshold randomly down-sample high-frequency words
                    hs=1, # hierarchical softmax instead of negative sampling
                    max_vocab_size=None, # no limit
                    alpha=0.025, # initial learning rate
                    min_alpha=0.001, # min learning rate
                    dm=0, # PV-DBOW 
                    dbow_words=0, # only trains doc-vectors
                    dm_tag_count=1, # one tag per document
                    dm_mean=0, # use the sum of the context word vectors
                    dm_concat=0, # for smaller model
                    negative=5, # number of negative samples
                    seed=seed, # random seed
                    workers=os.cpu_count()
                    )
    
    print("Building vocabulary")

    sentences = []
    for idx, row in tqdm(train_df.iterrows(), total=train_df.shape[0]):
        words_list = row[text_col].split()
        sentences.append(TaggedDocument(words_list, [row[target_col]]))

    print(sentences[:3])

    model.build_vocab(sentences)

    print("Training model")
    model.train(sentences, total_examples=model.corpus_count, epochs=num_epochs)

    print("Evaluating model")

    df_scored, metrics = evaluate_df_d2v(
        val_df,
        model=model,
        model_name=model_name,
        text_col=text_col,
        target_col=target_col,
        top_n=5,
        epochs=num_epochs,           # matches your trained-model naming
        alpha=None,          # let gensim handle the schedule
        min_alpha=None,
        show_progress=True
    )

    scored_dfs[model_name] = df_scored
    row = {'model': model_name, **metrics}
    all_metrics.append(row)

    del model

print("=== Metrics summary ===")
metrics_df = pd.DataFrame(all_metrics).set_index('model').sort_index()
display(metrics_df)
display(metrics_df.describe())

metrics_df.to_csv(os.path.join(out_dir, f"scored_dfs_{model_name}.csv"), index=False)
print(f"Saved metrics to {os.path.join(out_dir, f'scored_dfs_{model_name}.csv')}")

joblib.dump(
    scored_dfs,
    os.path.join(out_dir, f"scored_dfs_{model_name}.joblib"),
    compress=3
)
print(f"Saved scored_dfs to {os.path.join(out_dir, f'scored_dfs_{model_name}.joblib')}")


=== Iteration 1/10 seed 226944881 ===
Model name: D2V_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed226944881
Training Doc2Vec model
Building vocabulary


100%|██████████| 254721/254721 [00:07<00:00, 35938.01it/s]


[TaggedDocument(words=['brake', 'fluid', 'dot', '4', '50x200ml', 'brake', 'fluid', 'dot', 'x', 'ml', 'brake', 'fluid', 'dot', '4', '50x200ml', 'brake', 'fluid', 'dot', '4', '50x200ml', 'brake_fluid', 'fluid_dot', 'dot_4', '4_50x200ml', '50x200ml_brake', 'brake_fluid', 'fluid_dot', 'dot_x', 'x_ml', 'ml_brake', 'brake_fluid', 'fluid_dot', 'dot_4', '4_50x200ml', '50x200ml_brake', 'brake_fluid', 'fluid_dot', 'dot_4', '4_50x200ml'], tags=['2710']), TaggedDocument(words=['lcd', 'assembly', 'lcd', 'assembly', 'lcd', 'assembly', 'lcd', 'assembly', 'lcd_assembly', 'assembly_lcd', 'lcd_assembly', 'assembly_lcd', 'lcd_assembly', 'assembly_lcd', 'lcd_assembly'], tags=['8443']), TaggedDocument(words=['bearing', '22238', 'kcaw33c3', 'brand', 'mcb', 'bearing', 'kcaw', 'c', 'brand', 'mcb', 'bearing', '22238', 'kcaw33c3', 'brand', 'mcb', 'bearing', '22238', 'kcaw33c3', 'brand', 'mcb', 'bearing_22238', '22238_kcaw33c3', 'kcaw33c3_brand', 'brand_mcb', 'mcb_bearing', 'bearing_kcaw', 'kcaw_c', 'c_brand', '

100%|██████████| 13389/13389 [01:38<00:00, 135.33it/s]


Total samples: 13389
Top-1 Accuracy: 0.6095 (8160/13389)
Top-2 Accuracy: 0.6846 (9166/13389)
Top-3 Accuracy: 0.7135 (9552/13389)
Top-4 Accuracy: 0.7286 (9754/13389)
Top-5 Accuracy: 0.7417 (9930/13389)

=== Iteration 2/10 seed 768593320 ===
Model name: D2V_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed768593320
Training Doc2Vec model
Building vocabulary


...

100%|██████████| 13389/13389 [01:31<00:00, 146.66it/s]


Total samples: 13389
Top-1 Accuracy: 0.5959 (7977/13389)
Top-2 Accuracy: 0.6739 (9022/13389)
Top-3 Accuracy: 0.7080 (9480/13389)
Top-4 Accuracy: 0.7263 (9725/13389)
Top-5 Accuracy: 0.7393 (9898/13389)
=== Metrics summary ===


,top_1_acc,top_2_acc,top_3_acc,top_4_acc,top_5_acc
model,,,,,
D2V_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed173892995,0.621256,0.695496,0.723803,0.739712,0.751139
D2V_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed226944881,0.609530,0.684592,0.713496,0.728583,0.741654
D2V_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed366261559,0.609231,0.686459,0.715363,0.732168,0.743894
D2V_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed370247453,0.606916,0.689372,0.718724,0.736276,0.746956
D2V_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed503070489,0.595862,0.673911,0.708044,0.726343,0.739264
D2V_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed531210901,0.610576,0.689820,0.717156,0.733064,0.745463
D2V_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed625333247,0.598103,0.676003,0.707073,0.723878,0.735156
D2V_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed715275432,0.612070,0.685936,0.715438,0.732243,0.742326
D2V_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed768593320,0.612891,0.685413,0.716633,0.733886,0.744940


,top_1_acc,top_2_acc,top_3_acc,top_4_acc,top_5_acc
count,10.000000,10.000000,10.000000,10.000000,10.000000
mean,0.608432,0.684756,0.714870,0.731481,0.743080
std,0.007236,0.006488,0.004904,0.004722,0.004462
min,0.595862,0.673911,0.707073,0.723878,0.735156
25%,0.607159,0.681567,0.713104,0.728602,0.740421
50%,0.609380,0.685674,0.715400,0.732206,0.743110
75%,0.611696,0.688644,0.717025,0.733681,0.745332
max,0.621256,0.695496,0.723803,0.739712,0.751139


Saved metrics to results/baselines\scored_dfs_D2V_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed503070489.csv
Saved scored_dfs to results/baselines\scored_dfs_D2V_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed503070489.joblib


Took 204 min

## Training a FastText as baseline

A- Raw descriptions

B- Preproced descriptions

C- Preproced + N-gram descriptions

In [21]:
target_col = 'HS04'
window = 5 # context window size +/- 
num_epochs = 50
model_dim = 254
seed = 32

raw_col = 'GOODS_DESCRIPTION'
prepro_col = 'PREPRO_DESCRIPTION'
ngram_col = 'NGRAM_DESCRIPTION'

#### A- Raw descriptions

In [22]:
all_metrics = []
scored_dfs = {}

text_col = raw_col 

for iter in range(iterations):
    seed = seeds[iter]
    print(f"\n=== Iteration {iter+1}/{iterations} seed {seed} ===")
    model_name = f"FT_{text_col}_{target_col}_seed{seed}"
    print(f"Model name: {model_name}")

    train_df, val_df = bootstrap_sampling(df, test_fraction=fraction, seed=seed)

    print("Training FastText model")
    model = FastTextDocVec(dim=model_dim, 
                        window=window, 
                        min_count=1, # ignore 1 instead of not ignore any words 
                        epochs=num_epochs, 
                        sg=1, 
                        min_n=3, 
                        max_n=6)

    print("Training model")
    model.fit(train_df, text_col=text_col, label_col=target_col) 
    
    print("Evaluating model")

    df_scored, metrics = evaluate_df_ft(
        val_df,
        model=model,
        model_name=model_name,
        text_col=text_col,
        target_col=target_col,
        top_n=5,
        epochs=num_epochs,           # matches your trained-model naming
        alpha=None,          # let gensim handle the schedule
        min_alpha=None,
        show_progress=True
    )

    scored_dfs[model_name] = df_scored
    row = {'model': model_name, **metrics}
    all_metrics.append(row)

    del model

print("=== Metrics summary ===")
metrics_df = pd.DataFrame(all_metrics).set_index('model').sort_index()
display(metrics_df)
display(metrics_df.describe())

metrics_df.to_csv(os.path.join(out_dir, f"scored_dfs_{model_name}.csv"), index=False)
print(f"Saved metrics to {os.path.join(out_dir, f'scored_dfs_{model_name}.csv')}")

joblib.dump(
    scored_dfs,
    os.path.join(out_dir, f"scored_dfs_{model_name}.joblib"),
    compress=3
)
print(f"Saved scored_dfs to {os.path.join(out_dir, f'scored_dfs_{model_name}.joblib')}")


=== Iteration 1/10 seed 226944881 ===
Model name: FT_GOODS_DESCRIPTION_HS04_seed226944881
Training FastText model
Training model
Evaluating model
Model: FT_GOODS_DESCRIPTION_HS04_seed226944881
Text column: GOODS_DESCRIPTION
Target column: HS04
Top-N: 5


100%|██████████| 13389/13389 [20:46<00:00, 10.74it/s]


Total samples: 13389
Top-1 Accuracy: 0.5351 (7165/13389)
Top-2 Accuracy: 0.6590 (8823/13389)
Top-3 Accuracy: 0.7175 (9606/13389)
Top-4 Accuracy: 0.7520 (10068/13389)
Top-5 Accuracy: 0.7756 (10383/13389)

=== Iteration 2/10 seed 768593320 ===
Model name: FT_GOODS_DESCRIPTION_HS04_seed768593320
Training FastText model
Training model
Evaluating model
Model: FT_GOODS_DESCRIPTION_HS04_seed768593320
Text column: GOODS_DESCRIPTION
Target column: HS04
Top-N: 5


100%|██████████| 13389/13389 [20:47<00:00, 10.73it/s]


Total samples: 13389
Top-1 Accuracy: 0.5378 (7201/13389)
Top-2 Accuracy: 0.6641 (8891/13389)
Top-3 Accuracy: 0.7221 (9667/13389)
Top-4 Accuracy: 0.7537 (10090/13389)
Top-5 Accuracy: 0.7768 (10399/13389)

=== Iteration 3/10 seed 366261559 ===
Model name: FT_GOODS_DESCRIPTION_HS04_seed366261559
Training FastText model
Training model
Evaluating model
Model: FT_GOODS_DESCRIPTION_HS04_seed366261559
Text column: GOODS_DESCRIPTION
Target column: HS04
Top-N: 5


...

100%|██████████| 13389/13389 [20:55<00:00, 10.67it/s]

Total samples: 13389
Top-1 Accuracy: 0.5340 (7149/13389)
Top-2 Accuracy: 0.6567 (8792/13389)
Top-3 Accuracy: 0.7175 (9606/13389)
Top-4 Accuracy: 0.7512 (10058/13389)
Top-5 Accuracy: 0.7731 (10350/13389)
=== Metrics summary ===


,top_1_acc,top_2_acc,top_3_acc,top_4_acc,top_5_acc
model,,,,,
FT_GOODS_DESCRIPTION_HS04_seed173892995,0.547987,0.672642,0.730525,0.763089,0.784599
FT_GOODS_DESCRIPTION_HS04_seed226944881,0.535141,0.659048,0.717455,0.751961,0.775562
FT_GOODS_DESCRIPTION_HS04_seed366261559,0.539921,0.663455,0.719172,0.755322,0.779147
FT_GOODS_DESCRIPTION_HS04_seed370247453,0.533274,0.659347,0.717529,0.754873,0.778624
FT_GOODS_DESCRIPTION_HS04_seed503070489,0.534020,0.656733,0.717529,0.751214,0.773097
FT_GOODS_DESCRIPTION_HS04_seed531210901,0.538203,0.663306,0.718052,0.752185,0.777056
FT_GOODS_DESCRIPTION_HS04_seed625333247,0.533572,0.659497,0.716260,0.750317,0.775786
FT_GOODS_DESCRIPTION_HS04_seed715275432,0.530734,0.656434,0.711778,0.748824,0.773695
FT_GOODS_DESCRIPTION_HS04_seed768593320,0.537830,0.664127,0.722085,0.753678,0.776757


,top_1_acc,top_2_acc,top_3_acc,top_4_acc,top_5_acc
count,10.000000,10.000000,10.000000,10.000000,10.000000
mean,0.536993,0.661453,0.718762,0.753716,0.777272
std,0.004873,0.004756,0.004856,0.003984,0.003265
min,0.530734,0.656434,0.711778,0.748824,0.773097
25%,0.533684,0.659123,0.717287,0.751401,0.775618
50%,0.536485,0.659721,0.717529,0.752931,0.776906
75%,0.538987,0.663418,0.718892,0.755210,0.778568
max,0.547987,0.672642,0.730525,0.763089,0.784599


Saved metrics to results/baselines\scored_dfs_FT_GOODS_DESCRIPTION_HS04_seed503070489.csv
Saved scored_dfs to results/baselines\scored_dfs_FT_GOODS_DESCRIPTION_HS04_seed503070489.joblib


Took 234 min

#### B- Preproced descriptions

In [23]:
all_metrics = []
scored_dfs = {}

text_col = prepro_col 

for iter in range(iterations):
    seed = seeds[iter]
    print(f"\n=== Iteration {iter+1}/{iterations} seed {seed} ===")
    model_name = f"FT_{text_col}_{target_col}_seed{seed}"
    print(f"Model name: {model_name}")

    train_df, val_df = bootstrap_sampling(df, test_fraction=fraction, seed=seed)

    print("Training FastText model")
    model = FastTextDocVec(dim=model_dim, 
                        window=window, 
                        min_count=1, # ignore 1 instead of not ignore any words 
                        epochs=num_epochs, 
                        sg=1, 
                        min_n=3, 
                        max_n=6)

    print("Training model")
    model.fit(train_df, text_col=text_col, label_col=target_col) 
    
    print("Evaluating model")

    df_scored, metrics = evaluate_df_ft(
        val_df,
        model=model,
        model_name=model_name,
        text_col=text_col,
        target_col=target_col,
        top_n=5,
        epochs=num_epochs,           # matches your trained-model naming
        alpha=None,          # let gensim handle the schedule
        min_alpha=None,
        show_progress=True
    )

    scored_dfs[model_name] = df_scored
    row = {'model': model_name, **metrics}
    all_metrics.append(row)

    del model

print("=== Metrics summary ===")
metrics_df = pd.DataFrame(all_metrics).set_index('model').sort_index()
display(metrics_df)
display(metrics_df.describe())

metrics_df.to_csv(os.path.join(out_dir, f"scored_dfs_{model_name}.csv"), index=False)
print(f"Saved metrics to {os.path.join(out_dir, f'scored_dfs_{model_name}.csv')}")

joblib.dump(
    scored_dfs,
    os.path.join(out_dir, f"scored_dfs_{model_name}.joblib"),
    compress=3
)
print(f"Saved scored_dfs to {os.path.join(out_dir, f'scored_dfs_{model_name}.joblib')}")


=== Iteration 1/10 seed 226944881 ===
Model name: FT_PREPRO_DESCRIPTION_HS04_seed226944881
Training FastText model
Training model
Evaluating model
Model: FT_PREPRO_DESCRIPTION_HS04_seed226944881
Text column: PREPRO_DESCRIPTION
Target column: HS04
Top-N: 5


100%|██████████| 13389/13389 [21:12<00:00, 10.52it/s]


Total samples: 13389
Top-1 Accuracy: 0.5759 (7711/13389)
Top-2 Accuracy: 0.7028 (9410/13389)
Top-3 Accuracy: 0.7626 (10211/13389)
Top-4 Accuracy: 0.7963 (10661/13389)
Top-5 Accuracy: 0.8191 (10966/13389)

=== Iteration 2/10 seed 768593320 ===
Model name: FT_PREPRO_DESCRIPTION_HS04_seed768593320
Training FastText model
Training model
Evaluating model
Model: FT_PREPRO_DESCRIPTION_HS04_seed768593320
Text column: PREPRO_DESCRIPTION
Target column: HS04
Top-N: 5


100%|██████████| 13389/13389 [20:53<00:00, 10.69it/s]


Total samples: 13389
Top-1 Accuracy: 0.5772 (7727/13389)
Top-2 Accuracy: 0.7060 (9452/13389)
Top-3 Accuracy: 0.7671 (10270/13389)
Top-4 Accuracy: 0.8021 (10738/13389)
Top-5 Accuracy: 0.8239 (11030/13389)

=== Iteration 3/10 seed 366261559 ===
Model name: FT_PREPRO_DESCRIPTION_HS04_seed366261559
Training FastText model
Training model
Evaluating model
Model: FT_PREPRO_DESCRIPTION_HS04_seed366261559
Text column: PREPRO_DESCRIPTION
Target column: HS04
Top-N: 5


...

100%|██████████| 13389/13389 [20:35<00:00, 10.83it/s]

Total samples: 13389
Top-1 Accuracy: 0.5685 (7611/13389)
Top-2 Accuracy: 0.7005 (9378/13389)
Top-3 Accuracy: 0.7581 (10150/13389)
Top-4 Accuracy: 0.7907 (10587/13389)
Top-5 Accuracy: 0.8157 (10921/13389)
=== Metrics summary ===


,top_1_acc,top_2_acc,top_3_acc,top_4_acc,top_5_acc
model,,,,,
FT_PREPRO_DESCRIPTION_HS04_seed173892995,0.586377,0.715065,0.771454,0.804093,0.827918
FT_PREPRO_DESCRIPTION_HS04_seed226944881,0.575921,0.702816,0.762641,0.796251,0.819105
FT_PREPRO_DESCRIPTION_HS04_seed366261559,0.576518,0.707297,0.767496,0.800284,0.821495
FT_PREPRO_DESCRIPTION_HS04_seed370247453,0.576145,0.706475,0.764060,0.800060,0.822093
FT_PREPRO_DESCRIPTION_HS04_seed503070489,0.568452,0.700500,0.758085,0.790724,0.815670
FT_PREPRO_DESCRIPTION_HS04_seed531210901,0.581298,0.709538,0.761969,0.796923,0.818284
FT_PREPRO_DESCRIPTION_HS04_seed625333247,0.574800,0.702965,0.760102,0.794533,0.816939
FT_PREPRO_DESCRIPTION_HS04_seed715275432,0.569199,0.699828,0.758683,0.794608,0.816641
FT_PREPRO_DESCRIPTION_HS04_seed768593320,0.577190,0.705953,0.767122,0.802076,0.823885


,top_1_acc,top_2_acc,top_3_acc,top_4_acc,top_5_acc
count,10.000000,10.000000,10.000000,10.000000,10.000000
mean,0.576563,0.705587,0.763664,0.797961,0.820487
std,0.005301,0.004508,0.004241,0.004057,0.003848
min,0.568452,0.699828,0.758085,0.790724,0.815670
25%,0.575080,0.702853,0.760569,0.795019,0.817275
50%,0.576331,0.705692,0.763351,0.798492,0.820300
75%,0.579095,0.707091,0.766599,0.800228,0.822653
max,0.586377,0.715065,0.771454,0.804093,0.827918


Saved metrics to results/baselines\scored_dfs_FT_PREPRO_DESCRIPTION_HS04_seed503070489.csv
Saved scored_dfs to results/baselines\scored_dfs_FT_PREPRO_DESCRIPTION_HS04_seed503070489.joblib


Took 302 min

#### C- Preproced + N-gram descriptions

In [24]:
all_metrics = []
scored_dfs = {}

text_col = prepro_col +'_'+ ngram_col
df[text_col] = df[prepro_col] + ' ' + df[ngram_col]

for iter in range(iterations):
    seed = seeds[iter]
    print(f"\n=== Iteration {iter+1}/{iterations} seed {seed} ===")
    model_name = f"FT_{text_col}_{target_col}_seed{seed}"
    print(f"Model name: {model_name}")

    train_df, val_df = bootstrap_sampling(df, test_fraction=fraction, seed=seed)

    print("Training FastText model")
    model = FastTextDocVec(dim=model_dim, 
                        window=window, 
                        min_count=1, # ignore 1 instead of not ignore any words 
                        epochs=num_epochs, 
                        sg=1, 
                        min_n=3, 
                        max_n=6)

    print("Training model")
    model.fit(train_df, text_col=text_col, label_col=target_col) 
    
    print("Evaluating model")

    df_scored, metrics = evaluate_df_ft(
        val_df,
        model=model,
        model_name=model_name,
        text_col=text_col,
        target_col=target_col,
        top_n=5,
        epochs=num_epochs,           # matches your trained-model naming
        alpha=None,          # let gensim handle the schedule
        min_alpha=None,
        show_progress=True
    )

    scored_dfs[model_name] = df_scored
    row = {'model': model_name, **metrics}
    all_metrics.append(row)

    del model

print("=== Metrics summary ===")
metrics_df = pd.DataFrame(all_metrics).set_index('model').sort_index()
display(metrics_df)
display(metrics_df.describe())

metrics_df.to_csv(os.path.join(out_dir, f"scored_dfs_{model_name}.csv"), index=False)
print(f"Saved metrics to {os.path.join(out_dir, f'scored_dfs_{model_name}.csv')}")

joblib.dump(
    scored_dfs,
    os.path.join(out_dir, f"scored_dfs_{model_name}.joblib"),
    compress=3
)
print(f"Saved scored_dfs to {os.path.join(out_dir, f'scored_dfs_{model_name}.joblib')}")


=== Iteration 1/10 seed 226944881 ===
Model name: FT_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed226944881
Training FastText model
Training model
Evaluating model
Model: FT_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed226944881
Text column: PREPRO_DESCRIPTION_NGRAM_DESCRIPTION
Target column: HS04
Top-N: 5


100%|██████████| 13389/13389 [21:01<00:00, 10.61it/s]


Total samples: 13389
Top-1 Accuracy: 0.5829 (7804/13389)
Top-2 Accuracy: 0.7050 (9439/13389)
Top-3 Accuracy: 0.7606 (10183/13389)
Top-4 Accuracy: 0.7942 (10633/13389)
Top-5 Accuracy: 0.8179 (10950/13389)

=== Iteration 2/10 seed 768593320 ===
Model name: FT_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed768593320
Training FastText model
Training model
Evaluating model
Model: FT_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed768593320
Text column: PREPRO_DESCRIPTION_NGRAM_DESCRIPTION
Target column: HS04
Top-N: 5


100%|██████████| 13389/13389 [21:06<00:00, 10.58it/s]


Total samples: 13389
Top-1 Accuracy: 0.5781 (7739/13389)
Top-2 Accuracy: 0.7105 (9513/13389)
Top-3 Accuracy: 0.7676 (10276/13389)
Top-4 Accuracy: 0.7999 (10709/13389)
Top-5 Accuracy: 0.8245 (11039/13389)

=== Iteration 3/10 seed 366261559 ===
Model name: FT_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed366261559
Training FastText model
Training model
Evaluating model
Model: FT_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed366261559
Text column: PREPRO_DESCRIPTION_NGRAM_DESCRIPTION
Target column: HS04
Top-N: 5


...

100%|██████████| 13389/13389 [21:48<00:00, 10.23it/s]

Total samples: 13389
Top-1 Accuracy: 0.5714 (7650/13389)
Top-2 Accuracy: 0.7001 (9374/13389)
Top-3 Accuracy: 0.7600 (10176/13389)
Top-4 Accuracy: 0.7949 (10642/13389)
Top-5 Accuracy: 0.8175 (10944/13389)
=== Metrics summary ===


,top_1_acc,top_2_acc,top_3_acc,top_4_acc,top_5_acc
model,,,,,
FT_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed173892995,0.583389,0.713496,0.770110,0.803495,0.826201
FT_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed226944881,0.582941,0.704982,0.760624,0.794234,0.817910
FT_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed366261559,0.581447,0.706998,0.764732,0.797595,0.818060
FT_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed370247453,0.577340,0.704011,0.763313,0.798118,0.820375
FT_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed503070489,0.571439,0.700127,0.760027,0.794906,0.817462
FT_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed531210901,0.583613,0.706401,0.761521,0.796848,0.821271
FT_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed625333247,0.573456,0.700426,0.759504,0.791844,0.814624
FT_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed715275432,0.571514,0.698857,0.756292,0.791247,0.816715
FT_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed768593320,0.578086,0.710509,0.767570,0.799910,0.824483


,top_1_acc,top_2_acc,top_3_acc,top_4_acc,top_5_acc
count,10.000000,10.000000,10.000000,10.000000,10.000000
mean,0.578437,0.705183,0.762760,0.796467,0.819680
std,0.004843,0.004626,0.004065,0.003673,0.003553
min,0.571439,0.698857,0.756292,0.791247,0.814624
25%,0.574427,0.701322,0.760176,0.794402,0.817574
50%,0.579618,0.705504,0.762417,0.796662,0.818882
75%,0.582568,0.706849,0.764527,0.797987,0.821047
max,0.583613,0.713496,0.770110,0.803495,0.826201


Saved metrics to results/baselines\scored_dfs_FT_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed503070489.csv
Saved scored_dfs to results/baselines\scored_dfs_FT_PREPRO_DESCRIPTION_NGRAM_DESCRIPTION_HS04_seed503070489.joblib


Took 530 min

### Total time

3 model families D2V
- 49 GOODS DESCRIPTION
- 74 PREPRO
- 204 PREPRO + NGRAM (only over 70 %)


3 model families FT
- 234 GOODS DESCRIPTION
- 302 PREPRO
- 530 PREPRO + NGRAM

## Future steps

- Coding training for baselines using CV or similar method

Applied these iterations to DistiltBERT training and eval